# Predicting Smartphone Addiction

[Kaggle Playground Series S6E8](https://www.kaggle.com/competitions/playground-series-s6e8/overview). Binary classification scored on ROC AUC.

Hyperparameters come from `tune.py`. The iteration log lives in `README.md`.

In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

## Load

In [ ]:
train_df = pd.read_csv('data/train.csv')

train_df.head()

In [ ]:
train_df.info()

## Explore

In [ ]:
CAT = ['gender', 'stress_level', 'academic_work_impact']
TARGET = 'addicted_label'

for col in CAT:
    print(col, train_df[col].unique())

In [ ]:
train_df[TARGET].value_counts(ascending=True)

## Prepare

In [ ]:
DROP = ['id']

X = train_df.drop(columns=DROP + [TARGET])
y = train_df[TARGET]

X.shape, y.shape

## Model

In [ ]:
# from tune.py, 40-trial optuna search on fold 0
PARAMS = {
    'objective': 'binary',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
    'learning_rate': 0.05,
    'n_estimators': 1929,
    'num_leaves': 80,
    'min_child_samples': 97,
    'colsample_bytree': 0.523690144273565,
    'subsample': 0.9114664754160735,
    'subsample_freq': 1,
    'reg_alpha': 1.8611240417497987,
    'reg_lambda': 1.658990591801076e-05,
    'max_bin': 509,
}

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT),
    ],
    remainder='passthrough',
)

pipeline = Pipeline(
    steps=[
        ('pre', preprocess),
        ('model', LGBMClassifier(**PARAMS)),
    ]
)

## Validation

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(X))
fold_aucs = []

for train_idx, val_idx in CV.split(X, y):
    fold = clone(pipeline).fit(X.iloc[train_idx], y.iloc[train_idx])
    oof[val_idx] = fold.predict_proba(X.iloc[val_idx])[:, 1]
    fold_aucs.append(roc_auc_score(y.iloc[val_idx], oof[val_idx]))

print(f'OOF ROC AUC: {roc_auc_score(y, oof):.5f}')
print(f'per-fold:    {np.mean(fold_aucs):.5f} +/- {np.std(fold_aucs):.5f}')
print(f'             {np.round(fold_aucs, 5)}')

## Fit

In [ ]:
pipeline.fit(X, y)

fi = pd.DataFrame({
    'feature': pipeline.named_steps['pre'].get_feature_names_out(),
    'importance': pipeline.named_steps['model'].feature_importances_,
}).sort_values('importance', ascending=False, ignore_index=True)

fi

## Submission

In [ ]:
test_df = pd.read_csv('data/test.csv')
sample = pd.read_csv('data/sample_submission.csv')

test_pred = pipeline.predict_proba(test_df.drop(columns=DROP))[:, 1]

submission = pd.DataFrame({'id': test_df['id'], 'addicted_label': test_pred})

assert (submission['id'].values == sample['id'].values).all(), 'id mismatch vs sample_submission'
assert submission['addicted_label'].notna().all(), 'missing values in submission'

submission.to_csv('submission.csv', index=False)
submission.head()